# 05：在远程 CPU 上运行你的电路

**你将学到：**云端作业在另一个环境运行，有自己的 ID、状态和结果；量子电路仍是同一个 FlagQuantum 电路。

我们将一个非对称电路提交到九鼎。这是 CPU 实验，不是加速卡或性能加速演示。

[English edition](../notebooks/05_jiuding_jobs.ipynb)

**学习目标：** 提交一个 CPU 作业、恢复其回执，并比较远程与本地量子态。

**先修要求：** 已完成 01。真实执行还需要九鼎账号、已验证的镜像／项目／队列及共享存储。仅做本地预演不能验证云服务。

学习路线见 [workshop 导航](../README.md)。


## 1. 准备 workshop 环境

直接运行此设置单元，无需修改。它会找到你的个人结果目录。活动环境提供凭据和任务镜像，不要在 notebook 单元中填写密钥。在自己的电脑上使用时，请参考[环境准备](../SETUP.md)。

中英文版使用同一用户的个人结果目录和回执；切换语言不会自动创建新任务。不同用户的凭据和共享目录仍由平台管理。


In [ ]:
from pathlib import Path
import os
import sys

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. 构造非对称电路并预测答案

在三比特电路中只对 q0 施加 X 门，理想结果为 `100`，概率为 1。先在本地检查。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import flagquantum as fq

circuit = fq.Circuit(3).x(0)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
local = fq.run(circuit)
labels = [format(index, "03b") for index in range(8)]
print("Local probabilities (q0 q1 q2):", dict(zip(labels, local.state.reshape(-1).abs().square().tolist())))


## 3. 选择远程 CPU

镜像提供任务中安装的软件。活动环境已预设镜像、项目和队列；在其他环境中，请使用你自己的九鼎配置。
**打开提交开关并运行提交单元，会创建真实 CPU 作业，消耗队列资源。**

**概念：** 任务在另一个容器中运行，而不是当前 notebook 内核。镜像决定安装的软件与版本，共享存储用于交换脚本和结果。初始化会自动定位个人工作目录，下方直接显示实际配置。

**预期输出：** CPU 态向量与本地非对称电路一致（q0=1，因此结果为 100）。`Restored job` 表示恢复已保存的任务 ID，不是又执行了一次。

**常见误解：** `os.getenv` 只读取配置，不会自动发现或创建镜像。公开示例默认本地预演（`False`）；活动部署须在配置并验证镜像、项目和队列后，将可见开关设为 `True`。仅有凭据不会自动开启提交。


In [ ]:
target = "jiuding:cpu"
image = os.getenv("JIUDING_IMAGE", "")
project = os.getenv("JIUDING_PROJECT", "")
queue = os.getenv("JIUDING_QUEUE")
submit_to_cloud = False

print("Job image:", image or "not configured")
print("Project:", project or "not configured")
print("Queue:", queue or "platform default")
print("Result directory:", OUTPUTS)
print("Real submission enabled:", submit_to_cloud)


### 确定回执位置（直接运行，无需修改）

此单元在个人持久化结果目录中确定回执位置，并检查旧版 workshop 是否留下状态不明的提交。保存、配置校验和匹配恢复由 SDK 的 `receipt=` 完成，请保留回执及 SDK 创建的配套记录。

旧版手动保存的回执须显式使用 `fq.restore_job(receipt_path)`，不会被自动接管；不要删除旧记录来强行再次提交。


In [ ]:
receipt_path = OUTPUTS / "jiuding-x0-three-qubits.receipt.json"
job = None
if submit_to_cloud:
    if not image or not project:
        raise RuntimeError("Ask your instructor to configure the Jiuding image and project; nothing submitted.")
    if not receipt_path.exists() and (OUTPUTS / "jiuding-x0-three-qubits.submission.json").exists():
        raise RuntimeError("An earlier workshop attempt has no receipt. Reconcile it on the platform; do not resubmit.")
print("Receipt path:", receipt_path.resolve())


## 4. 通过 FlagQuantum 提交

`fq.submit(..., receipt=receipt_path)` 首次提交并自动保存回执，匹配的重复运行会恢复同一任务而不再次提交；调用返回时不会等待执行完成。

配置不同、记录不完整或恢复失败时会报错，不会自动重提；无需手动计算摘要或调用 `job.save()`。确认旧任务状态后，新的实验应使用新的回执路径。


In [ ]:
job = None
if submit_to_cloud:
    job = fq.submit(
        circuit, target=target, image=image, project=project, queue=queue,
        receipt=receipt_path,
    )
    print("Job ID:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. 查询进度

这里执行一次只读查询。`queued` 表示排队，`running` 表示正在执行，`succeeded` 表示平台报告已完成；结果交付仍可能延迟或暂不可用。

**读懂输出：** `Job ID` 标识任务，不表示计算完成。匹配的重复运行在 SDK 恢复后打印相同 ID。只有取回结果，才能完成本地与远程对照。

**安全重复运行：** 使用同一持久化回执路径重复调用提交，SDK 会校验并恢复原任务。等待时也可只重跑结果单元；不要删除旧记录来重试状态不明的提交。改变电路、target 或 shots 时，先核对旧任务，再为新实验选择新的路径。


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. 短暂等待并读取结果

`job.wait()` 将轮询状态和获取结果组合起来。超时不会停止任务，可以稍后重新运行此单元。
也可以在状态为 `succeeded` 后调用 `job.result()`。
如果远程任务失败，不会用本地结果冒充远程结果。


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    import torch

    torch.testing.assert_close(result.state, local.state)
    print("Remote probabilities (q0 q1 q2):", dict(zip(labels, result.state.reshape(-1).abs().square().tolist())))
    print("The remote CPU calculation matches the local calculation.")


## 带回家使用

离开 Jupyter 后仍然使用相同的标准 API。首次调用提交真实任务并保存回执，使用同一持久化路径重复运行会恢复它。SDK 负责配置校验与防重复提交；请先在代码之外配置镜像、项目和队列。

```python
import os
import flagquantum as fq

circuit = fq.Circuit(3).x(0)
job = fq.submit(
    circuit, target="jiuding:cpu", image=os.environ["JIUDING_IMAGE"],
    project=os.environ["JIUDING_PROJECT"], queue=os.getenv("JIUDING_QUEUE"),
    receipt="x0-three-qubits.receipt.json",
)
print(job.id, job.status())
```

恢复已保存任务：`job = fq.restore_job("x0-three-qubits.receipt.json")`。完成后可查看 `job.result().state`。

**解释结果：**哪些调用负责提交、查询、等待和获取结果？远程运行一个小电路是否能证明性能加速？
完整 Python 程序和加速器选项见[云任务说明](../CLOUD_JOBS.md)。
